# Facial Feature Restoration via Hybrid GAN

A residual GAN pipeline for recovering high-fidelity facial features from social media compression artifacts. The system trains a U-Net generator paired with global and local discriminators under a composite loss: L1 reconstruction, adversarial, and VGG-based identity preservation.

**Components**
- `Config` / `Workspace` — hyperparameters and environment-agnostic filesystem management
- `DistortionEngine` / `FaceDataset` — synthetic degradation and paired training data
- `Generator` / `Discriminator` / `FeatureDiscriminator` — adversarial architecture
- `IdentityPreserver` — frozen VGG-16 perceptual feature extractor
- `RestorationEngine` — training step encapsulating generator and discriminator optimization


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as T
import torchvision.utils as vutils
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import cv2
import random
import logging
import kagglehub
from PIL import Image
from tqdm.auto import tqdm
from dataclasses import dataclass
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric
import lpips
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

## 1. Configuration & Workspace

`Config` holds all training hyperparameters as a frozen dataclass. `Workspace` resolves execution environment at runtime — mounting Google Drive on Colab or falling back to a local directory — and exposes uniform interfaces for checkpoint persistence and visual logging.

In [ ]:
@dataclass
class Config:
    resolution: int = 256
    batch_size: int = 16
    learning_rate: float = 0.0002
    betas: tuple = (0.5, 0.999)
    num_epochs: int = 200
    lambda_rec: float = 100.0
    lambda_adv_g: float = 1.0
    lambda_adv_l: float = 1.0
    lambda_id: float = 10.0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


class Workspace:
    def __init__(self, name="facial_restoration"):
        self.base = self._resolve(name)
        self.dirs = {
            'ckpt': self._ensure("checkpoints"),
            'viz':  self._ensure("visualizations"),
        }

    def _resolve(self, name):
        if os.path.exists('/content'):
            try:
                from google.colab import drive
                drive.mount('/content/drive')
                return os.path.join('/content/drive/MyDrive', name)
            except Exception:
                return os.path.join('/content', name)
        return os.path.abspath(name)

    def _ensure(self, sub):
        p = os.path.join(self.base, sub)
        os.makedirs(p, exist_ok=True)
        return p

    def persist(self, epoch, models, optimizers, metric):
        state = {
            'epoch':      epoch,
            'models':     {k: v.state_dict() for k, v in models.items()},
            'optimizers': {k: v.state_dict() for k, v in optimizers.items()},
            'metric':     metric,
        }
        torch.save(state, os.path.join(self.dirs['ckpt'], 'latest.pth'))
        if epoch % 10 == 0:
            torch.save(state, os.path.join(self.dirs['ckpt'], f'epoch_{epoch}.pth'))

    def log_visuals(self, epoch, distorted, restored, gt):
        samples = torch.cat([distorted[:4], restored[:4], gt[:4]], dim=0)
        vutils.save_image(
            samples,
            os.path.join(self.dirs['viz'], f'step_{epoch:03d}.png'),
            nrow=4,
            normalize=True,
        )

## 2. Distortion Pipeline & Dataset

`DistortionEngine` applies stochastic synthetic degradation — randomised JPEG compression (quality 10–50) and additive Gaussian noise — to simulate social media re-encoding artefacts. `FaceDataset` constructs paired samples on-the-fly: each item returns a degraded tensor alongside its clean ground truth.

In [ ]:
class DistortionEngine(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p

    def _apply_jpeg(self, x):
        img = (x.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        quality = random.randint(10, 50)
        _, enc = cv2.imencode('.jpg', img, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
        dec = cv2.cvtColor(cv2.imdecode(enc, 1), cv2.COLOR_BGR2RGB)
        return torch.from_numpy(dec).permute(2, 0, 1).float().unsqueeze(0).to(x.device) / 255.0

    def forward(self, x):
        out = x.clone()
        if random.random() < self.p:
            out = self._apply_jpeg(out)
        if random.random() < self.p:
            out = torch.clamp(out + torch.randn_like(out) * 0.05, 0, 1)
        return out


class FaceDataset(Dataset):
    def __init__(self, root, res=256):
        self.files = [
            os.path.join(r, f)
            for r, _, fs in os.walk(root)
            for f in fs
            if f.lower().endswith(('.jpg', '.png'))
        ]
        self.tf = T.Compose([T.Resize((res, res)), T.CenterCrop(res), T.ToTensor()])
        self.engine = DistortionEngine()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        gt = self.tf(Image.open(self.files[idx]).convert('RGB'))
        dist = self.engine(gt.unsqueeze(0)).squeeze(0)
        return dist, gt

## 3. Model Architecture

### Generator — 8-Level U-Net with Residual Output

The generator produces a residual map $R$ rather than a full RGB image. The final reconstruction is $\hat{x} = x_{dist} + R$, clamped to $[0, 1]$. This formulation biases the network toward high-frequency correction while leaving intact the low-frequency structure already present in the degraded input. Skip connections at every encoder level are concatenated with the corresponding decoder feature maps.

In [ ]:
class UNetBlock(nn.Module):
    def __init__(self, in_c, out_c, down=True, dropout=False):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False, padding_mode='reflect')
            if down else
            nn.ConvTranspose2d(in_c, out_c, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.2, True) if down else nn.ReLU(True),
        )
        self.dropout = nn.Dropout(0.5) if dropout else None

    def forward(self, x):
        x = self.conv(x)
        return self.dropout(x) if self.dropout else x


class Generator(nn.Module):
    def __init__(self, features=64):
        super().__init__()
        f = features
        self.initial = nn.Sequential(
            nn.Conv2d(3, f, 4, 2, 1, padding_mode='reflect'),
            nn.LeakyReLU(0.2),
        )
        self.d1 = UNetBlock(f,    f*2)
        self.d2 = UNetBlock(f*2,  f*4)
        self.d3 = UNetBlock(f*4,  f*8)
        self.d4 = UNetBlock(f*8,  f*8)
        self.d5 = UNetBlock(f*8,  f*8)
        self.d6 = UNetBlock(f*8,  f*8)
        self.bn = nn.Sequential(
            nn.Conv2d(f*8, f*8, 4, 2, 1, padding_mode='reflect'),
            nn.ReLU(),
        )
        self.u1 = UNetBlock(f*8,  f*8,  down=False, dropout=True)
        self.u2 = UNetBlock(f*16, f*8,  down=False, dropout=True)
        self.u3 = UNetBlock(f*16, f*8,  down=False, dropout=True)
        self.u4 = UNetBlock(f*16, f*8,  down=False)
        self.u5 = UNetBlock(f*16, f*4,  down=False)
        self.u6 = UNetBlock(f*8,  f*2,  down=False)
        self.u7 = UNetBlock(f*4,  f,    down=False)
        self.final = nn.Sequential(
            nn.ConvTranspose2d(f*2, 3, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        e1 = self.initial(x)
        e2 = self.d1(e1);  e3 = self.d2(e2);  e4 = self.d3(e3)
        e5 = self.d4(e4);  e6 = self.d5(e5);  e7 = self.d6(e6)
        b  = self.bn(e7)

        up1 = self.u1(b)
        up2 = self.u2(torch.cat([up1, e7], 1))
        up3 = self.u3(torch.cat([up2, e6], 1))
        up4 = self.u4(torch.cat([up3, e5], 1))
        up5 = self.u5(torch.cat([up4, e4], 1))
        up6 = self.u6(torch.cat([up5, e3], 1))
        up7 = self.u7(torch.cat([up6, e2], 1))

        residual = self.final(torch.cat([up7, e1], 1))
        return torch.clamp(x + residual, 0, 1)

### Discriminators

Two discriminators operate in parallel:
- **Global (`Discriminator`)** — PatchGAN architecture that scores the full 256×256 image for photorealistic global coherence.
- **Local (`FeatureDiscriminator`)** — operates on fixed facial region crops (eye and perioral zones) to enforce fine-grained texture fidelity.

`IdentityPreserver` extracts deep feature vectors from a frozen VGG-16 backbone. The cosine distance between generator output and ground truth in this feature space constitutes the identity loss $\mathcal{L}_{id}$.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, in_c=3, features=None):
        super().__init__()
        if features is None:
            features = [64, 128, 256, 512]
        layers = [
            nn.Sequential(
                nn.Conv2d(in_c, features[0], 4, 2, 1, padding_mode='reflect'),
                nn.LeakyReLU(0.2, True),
            )
        ]
        in_c = features[0]
        for f in features[1:]:
            stride = 1 if f == features[-1] else 2
            layers.append(nn.Sequential(
                nn.Conv2d(in_c, f, 4, stride, 1, bias=False, padding_mode='reflect'),
                nn.BatchNorm2d(f),
                nn.LeakyReLU(0.2, True),
            ))
            in_c = f
        layers.append(nn.Conv2d(in_c, 1, 4, 1, 1, padding_mode='reflect'))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)


class FeatureDiscriminator(nn.Module):
    def __init__(self, in_c=3):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(in_c, 64,  4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64,   128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(128, 1, 1),
        )

    def forward(self, x):
        return self.model(x).view(x.size(0), 1)


class IdentityPreserver(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        self.feat = vgg.features[:23].eval()
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.feat(x).view(x.size(0), -1)

## 4. Restoration Engine

`RestorationEngine.step` executes one full training iteration. The generator is optimised against a weighted composite of four losses:

$$\mathcal{L}_G = \lambda_{rec}\,\mathcal{L}_1 + \lambda_{adv,g}\,\mathcal{L}_{adv}^{global} + \lambda_{adv,l}\,\mathcal{L}_{adv}^{local} + \lambda_{id}\,\mathcal{L}_{id}$$

The discriminator update follows standard LSGAN: real targets at 1, generated targets at 0, losses averaged across global and local heads.

In [ ]:
class RestorationEngine:
    def __init__(self, models, opts, config):
        self.gen    = models['gen']
        self.disc_g = models['dg']
        self.disc_l = models['dl']
        self.opt_g  = opts['gen']
        self.opt_d  = opts['disc']
        self.id_model = IdentityPreserver().to(config.device)
        self.config   = config
        self.mse = nn.MSELoss()
        self.l1  = nn.L1Loss()

    def _extract_patches(self, x):
        return torch.cat([x[:, :, 100:164, 96:160], x[:, :, 150:214, 96:160]], dim=0)

    def step(self, dist, gt):
        self.opt_g.zero_grad()
        res = self.gen(dist)

        g_adv_g = self.mse(self.disc_g(res),                            torch.ones_like(self.disc_g(res)))
        g_adv_l = self.mse(self.disc_l(self._extract_patches(res)),     torch.ones_like(self.disc_l(self._extract_patches(res))))
        g_rec   = self.l1(res, gt)
        g_id    = 1.0 - F.cosine_similarity(self.id_model(res), self.id_model(gt)).mean()

        g_loss = (
            self.config.lambda_rec   * g_rec
            + self.config.lambda_adv_g * g_adv_g
            + self.config.lambda_adv_l * g_adv_l
            + self.config.lambda_id    * g_id
        )
        g_loss.backward()
        self.opt_g.step()

        self.opt_d.zero_grad()
        d_real_g = self.mse(self.disc_g(gt),                             torch.ones_like(self.disc_g(gt)))
        d_fake_g = self.mse(self.disc_g(res.detach()),                   torch.zeros_like(self.disc_g(res)))
        d_real_l = self.mse(self.disc_l(self._extract_patches(gt)),      torch.ones_like(self.disc_l(self._extract_patches(gt))))
        d_fake_l = self.mse(self.disc_l(self._extract_patches(res.detach())), torch.zeros_like(self.disc_l(self._extract_patches(res))))

        d_loss = (d_real_g + d_fake_g + d_real_l + d_fake_l) * 0.25
        d_loss.backward()
        self.opt_d.step()

        return {
            'G':    g_loss.item(),
            'D':    d_loss.item(),
            'PSNR': 20 * np.log10(1.0 / np.sqrt(g_rec.item() + 1e-8)),
        }

## 5. Training Pipeline

Initialises all components, downloads the dataset via KaggleHub, splits 90 / 10 into train and validation sets, and runs the training loop. Checkpoints are written every epoch (latest) and every 10th epoch (named). Validation visualisations — input / restored / ground truth grids — are saved after each epoch.

In [ ]:
config    = Config()
workspace = Workspace()

path    = kagglehub.dataset_download('tommykamaz/faces-dataset-small')
dataset = FaceDataset(path)

train_size          = int(0.9 * len(dataset))
train_ds, val_ds    = random_split(dataset, [train_size, len(dataset) - train_size])
loader              = DataLoader(train_ds, batch_size=config.batch_size, shuffle=True)
val_loader          = DataLoader(val_ds,   batch_size=config.batch_size)

gen = Generator().to(config.device)
dg  = Discriminator().to(config.device)
dl  = FeatureDiscriminator().to(config.device)

opt_g = optim.Adam(gen.parameters(),                         lr=config.learning_rate, betas=config.betas)
opt_d = optim.Adam(list(dg.parameters()) + list(dl.parameters()), lr=config.learning_rate, betas=config.betas)

engine = RestorationEngine(
    {'gen': gen, 'dg': dg, 'dl': dl},
    {'gen': opt_g, 'disc': opt_d},
    config,
)

for epoch in range(config.num_epochs):
    bar = tqdm(loader, desc=f'Epoch {epoch:>3d}/{config.num_epochs}')
    for dist, gt in bar:
        metrics = engine.step(dist.to(config.device), gt.to(config.device))
        bar.set_postfix(G=f"{metrics['G']:.4f}", PSNR=f"{metrics['PSNR']:.2f}")

    workspace.persist(
        epoch,
        {'gen': gen, 'dg': dg, 'dl': dl},
        {'gen': opt_g, 'disc': opt_d},
        metrics['PSNR'],
    )

    dist_val, gt_val = next(iter(val_loader))
    with torch.no_grad():
        restored_val = gen(dist_val.to(config.device))
    workspace.log_visuals(epoch, dist_val, restored_val.cpu(), gt_val)